# 🔍 Treinamento de Detector de Deepfake - EfficientNet-B0

Este notebook treina um modelo EfficientNet-B0 para detecção de deepfakes em imagens faciais.

**Ambiente recomendado:** Google Colab Pro (GPU A100 ou V100)

## Bases de Dados Públicas Disponíveis

| Dataset | Tamanho | Tipo | Link |
|---------|---------|------|------|
| **FaceForensics++** | 1000 vídeos reais + 4000 fake | Vídeos | [GitHub](https://github.com/ondyari/FaceForensics) |
| **Celeb-DF** | 590 reais + 5639 fake | Vídeos | [GitHub](https://github.com/yuezunli/celeb-deepfakeforensics) |
| **DFDC (Kaggle)** | 100k+ vídeos | Vídeos | [Kaggle](https://www.kaggle.com/c/deepfake-detection-challenge) |
| **140k Real and Fake Faces** | 70k reais + 70k fake | Imagens | [Kaggle](https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces) |
| **DeepFake Detection (Kaggle)** | 191k faces | Imagens | [Kaggle](https://www.kaggle.com/datasets/dagnelies/deepfake-faces) |

Neste notebook usaremos o dataset **140k Real and Fake Faces** por ser de fácil acesso e já estar em formato de imagens.

## 1. Setup do Ambiente

In [ ]:
# Verificar GPU disponível
!nvidia-smi

# Instalar dependências
!pip install -q timm albumentations kaggle wandb

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
from tqdm.auto import tqdm
from datetime import datetime

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Reprodutibilidade
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Download do Dataset

### Opção A: Dataset 140k Real and Fake Faces (Kaggle)

In [ ]:
# Configurar credenciais do Kaggle
# Faça upload do seu kaggle.json ou configure manualmente:

# from google.colab import files
# files.upload()  # Upload kaggle.json

# Ou configure manualmente:
# os.environ['KAGGLE_USERNAME'] = 'seu_username'
# os.environ['KAGGLE_KEY'] = 'sua_api_key'

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/ 2>/dev/null || echo 'Configure suas credenciais do Kaggle'
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
# Download do dataset 140k Real and Fake Faces
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces -p /content/data --unzip

# Estrutura esperada:
# /content/data/real_vs_fake/real-vs-fake/
#   ├── train/
#   │   ├── real/
#   │   └── fake/
#   ├── valid/
#   └── test/

In [ ]:
# Verificar estrutura do dataset
DATA_ROOT = Path('/content/data/real_vs_fake/real-vs-fake')

for split in ['train', 'valid', 'test']:
    split_path = DATA_ROOT / split
    if split_path.exists():
        real_count = len(list((split_path / 'real').glob('*.jpg')))
        fake_count = len(list((split_path / 'fake').glob('*.jpg')))
        print(f'{split}: {real_count} reais, {fake_count} fakes')

### Opção B: FaceForensics++ (requer registro)

Para usar FaceForensics++:
1. Acesse https://github.com/ondyari/FaceForensics
2. Preencha o formulário de acesso
3. Use o script de download fornecido

In [ ]:
# # Download FaceForensics++ (descomente se tiver acesso)
# !git clone https://github.com/ondyari/FaceForensics.git
# !python FaceForensics/dataset/download_dataset.py \
#     /content/faceforensics \
#     -d all \
#     -c c23 \
#     -t videos

## 3. Configurações do Treinamento

In [ ]:
class Config:
    # Paths
    DATA_ROOT = Path('/content/data/real_vs_fake/real-vs-fake')
    OUTPUT_DIR = Path('/content/output')
    
    # Model
    MODEL_NAME = 'efficientnet_b0'
    PRETRAINED = True
    NUM_CLASSES = 1  # Binary: sigmoid output
    
    # Training
    EPOCHS = 10
    BATCH_SIZE = 64  # Ajuste conforme GPU (A100: 128, V100/T4: 64)
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    
    # Image
    IMG_SIZE = 224
    
    # Augmentation
    USE_AMP = True  # Mixed precision
    NUM_WORKERS = 4
    
    # Early stopping
    PATIENCE = 3
    
config = Config()
config.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 4. Dataset e Augmentations

In [ ]:
def get_transforms(is_train=True):
    """Augmentations para treinamento e validação."""
    if is_train:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(p=0.3),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
            A.GaussianBlur(blur_limit=(3, 5), p=0.2),
            A.ImageCompression(quality_lower=60, quality_upper=100, p=0.3),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.3),
            A.CoarseDropout(max_holes=4, max_height=20, max_width=20, p=0.2),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])


class DeepfakeDataset(Dataset):
    """Dataset para imagens de deepfake."""
    
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image
        image = np.array(Image.open(img_path).convert('RGB'))
        
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, torch.tensor(label, dtype=torch.float32)

In [ ]:
def prepare_data(data_root):
    """Prepara os caminhos das imagens e labels."""
    data = {'train': {'paths': [], 'labels': []},
            'valid': {'paths': [], 'labels': []},
            'test': {'paths': [], 'labels': []}}
    
    for split in ['train', 'valid', 'test']:
        split_path = data_root / split
        
        # Real images (label = 0)
        real_path = split_path / 'real'
        if real_path.exists():
            for img in real_path.glob('*.jpg'):
                data[split]['paths'].append(str(img))
                data[split]['labels'].append(0)
        
        # Fake images (label = 1)
        fake_path = split_path / 'fake'
        if fake_path.exists():
            for img in fake_path.glob('*.jpg'):
                data[split]['paths'].append(str(img))
                data[split]['labels'].append(1)
        
        print(f'{split}: {len(data[split]["paths"])} imagens')
    
    return data

# Preparar dados
data = prepare_data(config.DATA_ROOT)

In [ ]:
# Criar DataLoaders
train_dataset = DeepfakeDataset(
    data['train']['paths'],
    data['train']['labels'],
    transform=get_transforms(is_train=True)
)

valid_dataset = DeepfakeDataset(
    data['valid']['paths'],
    data['valid']['labels'],
    transform=get_transforms(is_train=False)
)

test_dataset = DeepfakeDataset(
    data['test']['paths'],
    data['test']['labels'],
    transform=get_transforms(is_train=False)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print(f'Train batches: {len(train_loader)}')
print(f'Valid batches: {len(valid_loader)}')
print(f'Test batches: {len(test_loader)}')

## 5. Modelo EfficientNet-B0

In [ ]:
class DeepfakeDetector(nn.Module):
    """Modelo de detecção de deepfake baseado em EfficientNet-B0."""
    
    def __init__(self, model_name='efficientnet_b0', pretrained=True, num_classes=1):
        super().__init__()
        
        # Backbone EfficientNet
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0  # Remove classifier
        )
        
        # Obter dimensão do feature
        self.num_features = self.backbone.num_features
        
        # Classifier head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.num_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes)
        )
    
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits.squeeze(-1)
    
    def predict_proba(self, x):
        """Retorna probabilidade de ser fake."""
        with torch.no_grad():
            logits = self.forward(x)
            probs = torch.sigmoid(logits)
        return probs


# Criar modelo
model = DeepfakeDetector(
    model_name=config.MODEL_NAME,
    pretrained=config.PRETRAINED,
    num_classes=config.NUM_CLASSES
).to(device)

print(f'Modelo: {config.MODEL_NAME}')
print(f'Parâmetros: {sum(p.numel() for p in model.parameters()):,}')
print(f'Parâmetros treináveis: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 6. Training Loop

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    """Treina uma época."""
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision forward
        with autocast(enabled=config.USE_AMP):
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        # Backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        
        # Predictions
        preds = torch.sigmoid(outputs).detach().cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())
        
        pbar.set_postfix({'loss': loss.item()})
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_preds = np.array(all_preds)
    epoch_labels = np.array(all_labels)
    
    accuracy = accuracy_score(epoch_labels, epoch_preds > 0.5)
    auc = roc_auc_score(epoch_labels, epoch_preds)
    
    return epoch_loss, accuracy, auc


def validate_epoch(model, loader, criterion, device):
    """Valida uma época."""
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)
            
            with autocast(enabled=config.USE_AMP):
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            
            preds = torch.sigmoid(outputs).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    
    epoch_loss = running_loss / len(loader.dataset)
    epoch_preds = np.array(all_preds)
    epoch_labels = np.array(all_labels)
    
    accuracy = accuracy_score(epoch_labels, epoch_preds > 0.5)
    auc = roc_auc_score(epoch_labels, epoch_preds)
    
    return epoch_loss, accuracy, auc, epoch_preds, epoch_labels

In [ ]:
# Optimizer, Loss, Scheduler
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.EPOCHS)
scaler = GradScaler(enabled=config.USE_AMP)

# Training history
history = {
    'train_loss': [], 'train_acc': [], 'train_auc': [],
    'valid_loss': [], 'valid_acc': [], 'valid_auc': []
}

best_auc = 0.0
patience_counter = 0

print('Iniciando treinamento...')
print('=' * 60)

In [ ]:
# Training loop
for epoch in range(config.EPOCHS):
    print(f'\nÉpoca {epoch + 1}/{config.EPOCHS}')
    print('-' * 40)
    
    # Train
    train_loss, train_acc, train_auc = train_epoch(
        model, train_loader, criterion, optimizer, scaler, device
    )
    
    # Validate
    valid_loss, valid_acc, valid_auc, _, _ = validate_epoch(
        model, valid_loader, criterion, device
    )
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['train_auc'].append(train_auc)
    history['valid_loss'].append(valid_loss)
    history['valid_acc'].append(valid_acc)
    history['valid_auc'].append(valid_auc)
    
    print(f'Train - Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, AUC: {train_auc:.4f}')
    print(f'Valid - Loss: {valid_loss:.4f}, Acc: {valid_acc:.4f}, AUC: {valid_auc:.4f}')
    
    # Save best model
    if valid_auc > best_auc:
        best_auc = valid_auc
        patience_counter = 0
        
        # Save checkpoint
        checkpoint = {
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_auc': best_auc,
            'config': {
                'model_name': config.MODEL_NAME,
                'img_size': config.IMG_SIZE,
                'num_classes': config.NUM_CLASSES
            }
        }
        torch.save(checkpoint, config.OUTPUT_DIR / 'best_model.pth')
        print(f'✅ Novo melhor modelo salvo! AUC: {best_auc:.4f}')
    else:
        patience_counter += 1
        if patience_counter >= config.PATIENCE:
            print(f'\nEarly stopping após {epoch + 1} épocas')
            break

print('\n' + '=' * 60)
print(f'Treinamento concluído! Melhor AUC: {best_auc:.4f}')

## 7. Avaliação no Test Set

In [ ]:
# Carregar melhor modelo
checkpoint = torch.load(config.OUTPUT_DIR / 'best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Modelo carregado do epoch {checkpoint["epoch"]} com AUC {checkpoint["best_auc"]:.4f}')

# Avaliar no test set
test_loss, test_acc, test_auc, test_preds, test_labels = validate_epoch(
    model, test_loader, criterion, device
)

print(f'\n📊 Resultados no Test Set:')
print(f'Loss: {test_loss:.4f}')
print(f'Accuracy: {test_acc:.4f}')
print(f'AUC-ROC: {test_auc:.4f}')

# Classification report
print('\n' + classification_report(test_labels, test_preds > 0.5, target_names=['Real', 'Fake']))

In [ ]:
# Plotar histórico de treinamento
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['valid_loss'], label='Valid')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(history['train_acc'], label='Train')
axes[1].plot(history['valid_acc'], label='Valid')
axes[1].set_title('Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

# AUC
axes[2].plot(history['train_auc'], label='Train')
axes[2].plot(history['valid_auc'], label='Valid')
axes[2].set_title('AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].legend()

plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_history.png', dpi=150)
plt.show()

## 8. Exportar Modelo para Produção

In [ ]:
# Salvar apenas os pesos do modelo (para uso no sistema)
production_weights = {
    'model_state_dict': model.state_dict(),
    'model_name': config.MODEL_NAME,
    'img_size': config.IMG_SIZE,
    'num_classes': config.NUM_CLASSES,
    'test_auc': test_auc,
    'test_acc': test_acc,
    'trained_on': '140k-real-and-fake-faces',
    'created_at': datetime.now().isoformat()
}

# Nome do arquivo com versão
model_filename = f'efficientnet_b0_deepfake_v1.pth'
torch.save(production_weights, config.OUTPUT_DIR / model_filename)
print(f'✅ Modelo de produção salvo: {model_filename}')

# Verificar tamanho
model_size = (config.OUTPUT_DIR / model_filename).stat().st_size / (1024 * 1024)
print(f'📦 Tamanho do modelo: {model_size:.1f} MB')

In [ ]:
# Exportar para TorchScript (opcional, para deploy otimizado)
model.eval()
example_input = torch.randn(1, 3, config.IMG_SIZE, config.IMG_SIZE).to(device)

# Trace model
traced_model = torch.jit.trace(model, example_input)
traced_model.save(str(config.OUTPUT_DIR / 'efficientnet_b0_deepfake_v1_traced.pt'))
print('✅ Modelo TorchScript salvo!')

In [ ]:
# Download dos arquivos (Google Colab)
from google.colab import files

# Download modelo de produção
files.download(str(config.OUTPUT_DIR / model_filename))

# Opcional: download do TorchScript
# files.download(str(config.OUTPUT_DIR / 'efficientnet_b0_deepfake_v1_traced.pt'))

## 9. Teste de Inferência

In [ ]:
def predict_image(model, image_path, transform, device):
    """Faz predição em uma única imagem."""
    model.eval()
    
    # Carregar imagem
    image = np.array(Image.open(image_path).convert('RGB'))
    
    # Aplicar transformações
    augmented = transform(image=image)
    image_tensor = augmented['image'].unsqueeze(0).to(device)
    
    # Inferência
    with torch.no_grad():
        with autocast(enabled=config.USE_AMP):
            logits = model(image_tensor)
            prob_fake = torch.sigmoid(logits).item()
    
    return prob_fake

# Testar em algumas imagens do test set
test_transform = get_transforms(is_train=False)

print('Exemplos de predição:')
print('-' * 40)

for i in range(5):
    img_path = data['test']['paths'][i]
    true_label = data['test']['labels'][i]
    prob = predict_image(model, img_path, test_transform, device)
    
    label_str = 'FAKE' if true_label == 1 else 'REAL'
    pred_str = 'FAKE' if prob > 0.5 else 'REAL'
    status = '✅' if (prob > 0.5) == true_label else '❌'
    
    print(f'{status} Real: {label_str} | Pred: {pred_str} ({prob:.2%})')

## 10. Instruções para Deploy

### Como usar o modelo treinado no sistema:

1. **Download do arquivo** `efficientnet_b0_deepfake_v1.pth`

2. **Copie para o diretório do worker:**
   ```bash
   cp efficientnet_b0_deepfake_v1.pth /path/to/detectar/model/weights/
   ```

3. **Atualize o config do worker** para apontar para o novo modelo:
   ```python
   MODEL_PATH = './model/weights/efficientnet_b0_deepfake_v1.pth'
   ```

4. **Reinicie o worker:**
   ```bash
   rq worker deepfake --url redis://localhost:6379/0
   ```

### Métricas esperadas:
- **Accuracy:** ~90-95% (dependendo do dataset)
- **AUC-ROC:** ~0.95-0.98
- **Tempo de inferência:** ~50-100ms por imagem (GPU)